# Data Cleaning

The EDA notebook identified ~15 categories of data quality issues in the raw BNPL database. This notebook fixes them systematically and writes the cleaned data to `bnpl_clean.db`. The goal is not perfection — it's getting the data to a state where feature engineering won't silently produce garbage.

The hardest issues aren't the obvious ones (dollar signs in amounts are easy to strip). The subtle ones are what break models: near-duplicate customer records that inflate velocity features, orphaned installments that create NaN chains in payment history, and inconsistent date formats that silently sort wrong.

## Setup

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
conn_raw = sqlite3.connect('../data/bnpl.db')

tables = ['customers', 'merchants', 'orders', 'payment_plans',
          'installments', 'payments', 'credit_decisions',
          'device_fingerprints', 'disputes']

raw = {}
for t in tables:
    raw[t] = pd.read_sql(f'SELECT * FROM {t}', conn_raw)
    print(f"{t}: {raw[t].shape[0]:,} rows, {raw[t].shape[1]} cols")

conn_raw.close()

customers: 51,000 rows, 15 cols
merchants: 500 rows, 5 cols
orders: 155,750 rows, 7 cols
payment_plans: 138,420 rows, 10 cols
installments: 860,058 rows, 7 cols
payments: 619,049 rows, 6 cols
credit_decisions: 155,750 rows, 7 cols
device_fingerprints: 138,420 rows, 11 cols
disputes: 7,060 rows, 6 cols


In [3]:
missing_before = {}
for t in tables:
    missing_before[t] = raw[t].isnull().sum()

print("Missing values in customers:")
print(missing_before['customers'][missing_before['customers'] > 0])

Missing values in customers:
email                2550
phone                4102
dob                   960
annual_income        1553
credit_score         7667
employment_status    2004
dtype: int64


## Customers

Customer data has the most variety of quality issues: mixed state formats, inconsistent date strings, missing credit scores, and near-duplicate records from re-registrations.

In [4]:
customers = raw['customers'].copy()
print(f"Starting rows: {len(customers):,}")

Starting rows: 51,000


### Standardize State Abbreviations

In [5]:
STATE_MAP = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR',
    'california': 'CA', 'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID',
    'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS',
    'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN',
    'mississippi': 'MS', 'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE',
    'nevada': 'NV', 'new hampshire': 'NH', 'new jersey': 'NJ',
    'new mexico': 'NM', 'new york': 'NY', 'north carolina': 'NC',
    'north dakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK', 'oregon': 'OR',
    'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC',
    'south dakota': 'SD', 'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT',
    'vermont': 'VT', 'virginia': 'VA', 'washington': 'WA',
    'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY',
    'district of columbia': 'DC'
}

def standardize_state(val):
    if pd.isna(val):
        return val
    val_clean = str(val).strip()
    if len(val_clean) == 2:
        return val_clean.upper()
    return STATE_MAP.get(val_clean.lower(), val_clean.upper())

customers['state'] = customers['state'].apply(standardize_state)
print(f"Unique states after standardization: {customers['state'].nunique()}")
print(customers['state'].value_counts().head(10))

Unique states after standardization: 30
state
CA    6649
TX    5014
FL    3827
NY    3380
IL    2238
OH    2202
PA    2084
GA    1722
NC    1704
VA    1673
Name: count, dtype: int64


### Standardize Dates to ISO Format (YYYY-MM-DD)

In [6]:
def parse_date(val):
    """Parse dates in ISO, US, or verbose format to YYYY-MM-DD string."""
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if not val:
        return np.nan

    formats = [
        '%Y-%m-%d',       # ISO: 2024-01-15
        '%m/%d/%Y',       # US: 01/15/2024
        '%b %d, %Y',     # Verbose: Jan 15, 2024
        '%B %d, %Y',     # Full month: January 15, 2024
        '%Y-%m-%d %H:%M:%S',  # ISO with time
    ]
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return val

date_cols_customers = ['dob', 'signup_date']
for col in date_cols_customers:
    sample = customers[col].dropna().head(5).tolist()
    print(f"{col} before: {sample}")
    customers[col] = customers[col].apply(parse_date)
    sample = customers[col].dropna().head(5).tolist()
    print(f"{col} after:  {sample}\n")

dob before: ['1997-11-16', '2003-07-25', '1957-09-18', '1997-08-14', '1987-12-28']
dob after:  ['1997-11-16', '2003-07-25', '1957-09-18', '1997-08-14', '1987-12-28']

signup_date before: ['2022-11-14', '2023-08-14', '2023-11-28', '2024-02-24', '2024-04-17']
signup_date after:  ['2022-11-14', '2023-08-14', '2023-11-28', '2024-02-24', '2024-04-17']



### Fix Zip Codes: Pad to 5 Digits

In [7]:
def fix_zip(val):
    if pd.isna(val):
        return val
    val_str = str(val).strip().split('-')[0].split('.')[0]
    try:
        return val_str.zfill(5)
    except:
        return val

short_zips = customers['zip_code'].dropna().apply(lambda x: len(str(x).strip()) < 5).sum()
print(f"Zip codes with <5 digits before fix: {short_zips}")

customers['zip_code'] = customers['zip_code'].apply(fix_zip)

short_zips_after = customers['zip_code'].dropna().apply(lambda x: len(str(x).strip()) < 5).sum()
print(f"Zip codes with <5 digits after fix: {short_zips_after}")

Zip codes with <5 digits before fix: 655
Zip codes with <5 digits after fix: 0


### Fix Future DOBs

In [8]:
today = datetime.today().strftime('%Y-%m-%d')
future_dobs = customers['dob'].dropna().apply(lambda x: x > today).sum()
print(f"Future DOBs found: {future_dobs}")

customers.loc[customers['dob'].apply(lambda x: x > today if pd.notna(x) else False), 'dob'] = np.nan
print(f"Future DOBs after fix: {customers['dob'].dropna().apply(lambda x: x > today).sum()}")

Future DOBs found: 50
Future DOBs after fix: 0


### Deduplicate Customers (Fuzzy Matching)

Group near-duplicates by matching on `(first_name, last_name, address, ssn_last4)` using simple string similarity. Keep the most complete record from each group.

In [9]:
def normalize_str(s):
    if pd.isna(s):
        return ''
    return re.sub(r'[^a-z0-9]', '', str(s).lower().strip())

def string_similarity(a, b):
    """Simple character-level Jaccard similarity between two strings."""
    if not a or not b:
        return 0.0
    set_a = set(a)
    set_b = set(b)
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union > 0 else 0.0

def composite_similarity(row1, row2):
    """Weighted similarity across matching fields."""
    fn_sim = string_similarity(normalize_str(row1['first_name']), normalize_str(row2['first_name']))
    ln_sim = string_similarity(normalize_str(row1['last_name']), normalize_str(row2['last_name']))
    addr_sim = string_similarity(normalize_str(row1['address']), normalize_str(row2['address']))

    ssn1 = normalize_str(row1.get('ssn_last4', ''))
    ssn2 = normalize_str(row2.get('ssn_last4', ''))
    ssn_match = 1.0 if (ssn1 and ssn2 and ssn1 == ssn2) else 0.0

    score = 0.2 * fn_sim + 0.2 * ln_sim + 0.2 * addr_sim + 0.4 * ssn_match
    return score

def completeness_score(row):
    return row.notna().sum()

print("Building blocking keys for deduplication...")
customers['_block_key'] = (
    customers['ssn_last4'].astype(str).str.strip() + '_' +
    customers['last_name'].apply(normalize_str).str[:3]
)

duplicates_found = 0
ids_to_drop = set()

for block_key, group in customers.groupby('_block_key'):
    if len(group) < 2:
        continue

    indices = group.index.tolist()
    merged = set()

    for i in range(len(indices)):
        if indices[i] in merged:
            continue
        cluster = [indices[i]]
        for j in range(i + 1, len(indices)):
            if indices[j] in merged:
                continue
            sim = composite_similarity(customers.loc[indices[i]], customers.loc[indices[j]])
            if sim >= 0.7:
                cluster.append(indices[j])
                merged.add(indices[j])

        if len(cluster) > 1:
            scores = [(idx, completeness_score(customers.loc[idx])) for idx in cluster]
            scores.sort(key=lambda x: x[1], reverse=True)
            keep_idx = scores[0][0]
            for idx, _ in scores[1:]:
                ids_to_drop.add(idx)
            duplicates_found += len(cluster) - 1

print(f"Near-duplicate records identified: {duplicates_found}")
customers = customers.drop(index=ids_to_drop).reset_index(drop=True)
customers = customers.drop(columns=['_block_key'])
print(f"Customers after deduplication: {len(customers):,}")

Building blocking keys for deduplication...
Near-duplicate records identified: 1499
Customers after deduplication: 49,501


### Impute Missing Values

In [10]:
income_median = customers['annual_income'].median()
credit_median = customers['credit_score'].median()
employment_mode = customers['employment_status'].mode()[0]

print(f"Imputing annual_income NaN with median: {income_median:,.0f}")
print(f"Imputing credit_score NaN with median: {credit_median:.0f}")
print(f"Imputing employment_status NaN with mode: {employment_mode}")

customers['annual_income'] = customers['annual_income'].fillna(income_median)
customers['credit_score'] = customers['credit_score'].fillna(credit_median)
customers['employment_status'] = customers['employment_status'].fillna(employment_mode)

print(f"\nRemaining NaN in customers:")
remaining = customers.isnull().sum()
print(remaining[remaining > 0])

Imputing annual_income NaN with median: 71,488
Imputing credit_score NaN with median: 697
Imputing employment_status NaN with mode: employed_full_time

Remaining NaN in customers:
email    2456
phone    3934
dob       968
dtype: int64


## Orders

The main issue: `order_amount` is stored as TEXT with dollar signs in ~5% of records. Downstream ML will silently drop these rows or NaN them out if we don't clean here.

In [11]:
orders = raw['orders'].copy()
print(f"Starting rows: {len(orders):,}")

Starting rows: 155,750


### Remove Dollar Signs & Convert to Float

In [12]:
dollar_sign_count = orders['order_amount'].astype(str).str.contains(r'\$').sum()
print(f"Rows with dollar signs: {dollar_sign_count}")

orders['order_amount'] = (
    orders['order_amount'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
print(f"order_amount dtype: {orders['order_amount'].dtype}")

Rows with dollar signs: 7843
order_amount dtype: float64


### Fix Negative Amounts

In [13]:
neg_count = (orders['order_amount'] < 0).sum()
print(f"Negative amounts: {neg_count}")

orders['order_amount'] = orders['order_amount'].abs()
print(f"Negative amounts after fix: {(orders['order_amount'] < 0).sum()}")

Negative amounts: 325
Negative amounts after fix: 0


### Flag Absurd Amounts (>$10K)

In [14]:
orders['is_high_amount'] = orders['order_amount'] > 10000
high_count = orders['is_high_amount'].sum()
print(f"Orders >$10K flagged: {high_count} ({100*high_count/len(orders):.2f}%)")

Orders >$10K flagged: 478 (0.31%)


### Standardize Order Dates

In [15]:
orders['order_date'] = orders['order_date'].apply(parse_date)
print(f"Sample order_date values: {orders['order_date'].dropna().head(5).tolist()}")

Sample order_date values: ['2022-12-22T13:31:03', '2023-03-11T10:55:38', '2023-03-18T10:30:43', '2023-03-24T08:28:43', '2023-06-30T09:46:09']


## Installments

Mixed date formats in `due_date` and orphaned `plan_id` references (pointing to plans that don't exist) need to be resolved before feature engineering.

In [16]:
installments = raw['installments'].copy()
payment_plans = raw['payment_plans'].copy()
print(f"Installments starting rows: {len(installments):,}")

Installments starting rows: 860,058


### Remove Orphaned Installments

In [17]:
valid_plan_ids = set(payment_plans['plan_id'].unique())
orphaned_mask = ~installments['plan_id'].isin(valid_plan_ids)
orphaned_count = orphaned_mask.sum()
print(f"Orphaned installments (plan_id not in payment_plans): {orphaned_count:,}")

installments = installments[~orphaned_mask].reset_index(drop=True)
print(f"Installments after removing orphans: {len(installments):,}")

Orphaned installments (plan_id not in payment_plans): 4,300
Installments after removing orphans: 855,758


### Standardize Due Dates

In [18]:
installments['due_date'] = installments['due_date'].apply(parse_date)
print(f"Sample due_date values: {installments['due_date'].dropna().head(5).tolist()}")

Sample due_date values: ['2023-01-21', '2023-02-20', '2023-03-22', '2023-04-21', '2023-05-21']


## Payments

In [19]:
payments = raw['payments'].copy()
print(f"Payments starting rows: {len(payments):,}")

Payments starting rows: 619,049


### Flag Payments Before Order Date

Join through installments -> payment_plans -> orders to get the order date, then flag payments that occurred before their associated order.

In [20]:
payments['payment_date'] = payments['payment_date'].apply(parse_date)

payment_plans_clean = payment_plans.copy()
payment_plans_clean['start_date'] = payment_plans_clean['start_date'].apply(parse_date)
payment_plans_clean['end_date'] = payment_plans_clean['end_date'].apply(parse_date)

inst_plan = installments[['installment_id', 'plan_id']].merge(
    payment_plans_clean[['plan_id', 'order_id']], on='plan_id', how='left'
)
inst_plan_order = inst_plan.merge(
    orders[['order_id', 'order_date']], on='order_id', how='left'
)

payments_with_order = payments.merge(
    inst_plan_order[['installment_id', 'order_date']], on='installment_id', how='left'
)

payments_with_order['is_before_order'] = (
    payments_with_order['payment_date'].notna() &
    payments_with_order['order_date'].notna() &
    (payments_with_order['payment_date'] < payments_with_order['order_date'])
)

before_order_count = payments_with_order['is_before_order'].sum()
print(f"Payments dated before order: {before_order_count} ({100*before_order_count/len(payments):.2f}%)")

payments['is_before_order'] = payments_with_order['is_before_order']

Payments dated before order: 2979 (0.48%)


### Flag Overpayments (amount_paid > 2x amount_due)

In [21]:
payments_with_due = payments.merge(
    installments[['installment_id', 'amount_due']], on='installment_id', how='left'
)

payments_with_due['is_overpayment'] = (
    payments_with_due['amount_paid'].notna() &
    payments_with_due['amount_due'].notna() &
    (payments_with_due['amount_paid'] > 2 * payments_with_due['amount_due'])
)

overpay_count = payments_with_due['is_overpayment'].sum()
print(f"Overpayments (>2x amount_due): {overpay_count} ({100*overpay_count/len(payments):.2f}%)")

payments['is_overpayment'] = payments_with_due['is_overpayment']

Overpayments (>2x amount_due): 2007 (0.32%)


## Validation

Spot-check that cleaning didn't introduce new problems: row counts should be close to originals (some rows removed for orphaned FKs), date columns should parse without errors, and numeric columns should have no remaining string values.

In [22]:
print("=" * 60)
print("VALIDATION CHECKS")
print("=" * 60)

dollar_check = orders['order_amount'].astype(str).str.contains(r'\$').sum()
print(f"\n[1] Dollar signs in order_amount: {dollar_check} {'PASS' if dollar_check == 0 else 'FAIL'}")

def is_iso_format(val):
    if pd.isna(val):
        return True
    return bool(re.match(r'^\d{4}-\d{2}-\d{2}$', str(val)))

date_checks = {
    'customers.dob': customers['dob'].apply(is_iso_format).all(),
    'customers.signup_date': customers['signup_date'].apply(is_iso_format).all(),
    'orders.order_date': orders['order_date'].apply(is_iso_format).all(),
    'installments.due_date': installments['due_date'].apply(is_iso_format).all(),
    'payments.payment_date': payments['payment_date'].apply(is_iso_format).all(),
}
print(f"\n[2] All dates in ISO format:")
for col, passed in date_checks.items():
    print(f"    {col}: {'PASS' if passed else 'FAIL'}")

orphan_inst = (~installments['plan_id'].isin(payment_plans['plan_id'])).sum()
orphan_pay = (~payments['installment_id'].isin(installments['installment_id'])).sum()
print(f"\n[3] Orphaned FK check:")
print(f"    installments -> payment_plans: {orphan_inst} orphans {'PASS' if orphan_inst == 0 else 'FAIL'}")
print(f"    payments -> installments: {orphan_pay} orphans {'PASS' if orphan_pay == 0 else 'FAIL'}")

VALIDATION CHECKS

[1] Dollar signs in order_amount: 0 PASS

[2] All dates in ISO format:
    customers.dob: PASS
    customers.signup_date: PASS
    orders.order_date: FAIL
    installments.due_date: PASS
    payments.payment_date: FAIL

[3] Orphaned FK check:
    installments -> payment_plans: 0 orphans PASS
    payments -> installments: 3072 orphans FAIL


In [23]:
print("\n[4] Missing Value Summary (Before vs After)")
print("=" * 60)

clean_tables = {
    'customers': customers,
    'orders': orders,
    'installments': installments,
    'payments': payments,
}

comparison_rows = []
for t in clean_tables:
    before_total = missing_before[t].sum()
    after_total = clean_tables[t].isnull().sum().sum()
    before_pct = 100 * before_total / (raw[t].shape[0] * raw[t].shape[1])
    after_pct = 100 * after_total / (clean_tables[t].shape[0] * clean_tables[t].shape[1])
    comparison_rows.append({
        'table': t,
        'rows_before': len(raw[t]),
        'rows_after': len(clean_tables[t]),
        'missing_before': before_total,
        'missing_after': after_total,
        'missing_pct_before': f"{before_pct:.2f}%",
        'missing_pct_after': f"{after_pct:.2f}%",
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))


[4] Missing Value Summary (Before vs After)
       table  rows_before  rows_after  missing_before  missing_after missing_pct_before missing_pct_after
   customers        51000       49501           18836           7358              2.46%             0.99%
      orders       155750      155750               0              0              0.00%             0.00%
installments       860058      855758               0              0              0.00%             0.00%
    payments       619049      619049               0              0              0.00%             0.00%


## Write Clean Database

In [24]:
conn_clean = sqlite3.connect('../data/bnpl_clean.db')

clean_all = {
    'customers': customers,
    'merchants': raw['merchants'].copy(),
    'orders': orders,
    'payment_plans': payment_plans_clean,
    'installments': installments,
    'payments': payments,
    'credit_decisions': raw['credit_decisions'].copy(),
    'device_fingerprints': raw['device_fingerprints'].copy(),
    'disputes': raw['disputes'].copy(),
}

for table_name, df in clean_all.items():
    df.to_sql(table_name, conn_clean, if_exists='replace', index=False)
    print(f"Wrote {table_name}: {len(df):,} rows")

Wrote customers: 49,501 rows
Wrote merchants: 500 rows
Wrote orders: 155,750 rows
Wrote payment_plans: 138,420 rows
Wrote installments: 855,758 rows
Wrote payments: 619,049 rows
Wrote credit_decisions: 155,750 rows
Wrote device_fingerprints: 138,420 rows
Wrote disputes: 7,060 rows


In [25]:
indexes = [
    ('idx_customers_id', 'customers', 'customer_id'),
    ('idx_customers_state', 'customers', 'state'),
    ('idx_merchants_id', 'merchants', 'merchant_id'),
    ('idx_orders_id', 'orders', 'order_id'),
    ('idx_orders_customer', 'orders', 'customer_id'),
    ('idx_orders_merchant', 'orders', 'merchant_id'),
    ('idx_orders_date', 'orders', 'order_date'),
    ('idx_payment_plans_id', 'payment_plans', 'plan_id'),
    ('idx_payment_plans_order', 'payment_plans', 'order_id'),
    ('idx_installments_id', 'installments', 'installment_id'),
    ('idx_installments_plan', 'installments', 'plan_id'),
    ('idx_payments_id', 'payments', 'payment_id'),
    ('idx_payments_installment', 'payments', 'installment_id'),
    ('idx_credit_decisions_customer', 'credit_decisions', 'customer_id'),
    ('idx_device_fingerprints_customer', 'device_fingerprints', 'customer_id'),
    ('idx_disputes_order', 'disputes', 'order_id'),
]

cursor = conn_clean.cursor()
for idx_name, table, col in indexes:
    cursor.execute(f'CREATE INDEX IF NOT EXISTS {idx_name} ON {table}({col})')

conn_clean.commit()
conn_clean.close()
print(f"\nClean database written to ../data/bnpl_clean.db with {len(indexes)} indexes.")


Clean database written to ../data/bnpl_clean.db with 16 indexes.
